In [0]:
import json
import pandas as pd

# 1. Ruta del archivo en tu Drop Zone
RUTA_BCRP = "/Workspace/peru-trade-analytics/bcrp.json"

# Leemos el archivo JSON descargado manualmente
with open(RUTA_BCRP, "r", encoding="utf-8") as f:
    data = json.load(f)

In [0]:
# 2. Diccionario de meses y Parser adaptado para los puntos
MESES_ES = {
    "Ene": "01", "Feb": "02", "Mar": "03", "Abr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Ago": "08",
    "Set": "09", "Oct": "10", "Nov": "11", "Dic": "12",
}

def _parsear_fecha_bcrp(nombre: str) -> str:
    """Convierte el formato '02.Ene.24' a '2024-01-02' separando por el punto."""
    dia, mes_abr, anio2 = nombre.split('.')
    
    anio = f"20{anio2}" if int(anio2) < 50 else f"19{anio2}"
    mes = MESES_ES.get(mes_abr, "01")
    
    return f"{anio}-{mes}-{dia}"

In [0]:
# 3. Extracción filtrando fines de semana ("n.d.")
filas = []
for p in data.get("periods", []):
    valor = p["values"][0]
    if valor == "n.d.":
        continue 
    
    filas.append({
        "fecha": _parsear_fecha_bcrp(p["name"]), 
        "tasa_cambio": float(valor)
    })

df_bcrp = pd.DataFrame(filas)
print(f"Registros útiles extraídos (Pandas): {len(df_bcrp)}")

Registros útiles extraídos (Pandas): 2494


In [0]:
# 4. Convertir a motor distribuido (Spark DataFrame)
df_bcrp_spark = spark.createDataFrame(df_bcrp)

print(f"Registros en Spark: {df_bcrp_spark.count()}")

# Mostrar la tabla interactiva
display(df_bcrp_spark)

Registros en Spark: 2494


fecha,tasa_cambio
2015-01-05,2.992
2015-01-06,2.983
2015-01-07,2.987
2015-01-08,2.989
2015-01-09,2.986
2015-01-12,2.989
2015-01-13,2.985
2015-01-14,2.994
2015-01-15,2.997
2015-01-16,3.013


In [ ]:
# Guardamos la tabla en formato Delta (Capa Bronze)
df_bcrp_spark.write.format("delta").mode("overwrite").saveAsTable("bcrp_crudo")
print("¡Tabla BCRP guardada exitosamente en el catálogo!")